# 2D Global Map Visualization — P2SAoM40 Kernel Comparison (DRYCNV + PBL)

Similar in purpose and style to `mantle/visualize_2d_global_maps.ipynb`, but built on this
project's **real** Fortran-vs-JAX kernel comparison in `compare_data/` (see
`STATUS.md`) instead of the older synthetic single-layer
placeholder workflow. That older notebook currently can't run — it expects files
like `temperature_jax.npy`/`temperature_fortran.npy` in `outputs/` that were never
generated (only a bare `heat_flux.npy` exists there); this notebook uses data that
actually exists and was actually validated.

**Important — what's real here and what isn't:**
- **Real**: the grid shape (72 lon × 46 lat, P2SAoM40's actual medium-resolution
  grid) and the lat/lon axis coordinates (pulled directly from the archived
  production output `ANN4099.aijP2SAoM40.nc`), the Fortran reference (compiled with
  `ifort -O2`, the real ModelE toolchain), and the JAX CPU/GPU runs (JAX-GPU run
  2026-09-20 on a real GPU node).
- **Not real**: the field *values*. DRYCNV's T/Q/PK/PDSIG and PBL's z/z0m/.../qg
  are synthetic random test vectors (fixed seed, see `compare_generate_inputs.py`)
  sized to this grid — not actual simulated climate fields. Reshaping them onto
  the real grid shape below does **not** mean the maps show real weather; the
  point of this notebook is to look at *where* (if anywhere) JAX and Fortran
  diverge across the grid, as a spatial complement to the scalar accuracy table
  in `STATUS.md` — not to interpret the patterns as climate.

DRYCNV is inherently 3-D (72×46×40); this notebook shows a single representative
layer (index 0, the model's lowest/surface-most layer) for the "2D global map"
framing. PBL's fields (u, t, q, dpsim, dpsih, dpsiq) are naturally single-layer
surface/boundary-layer diagnostics, so no layer selection is needed for those.

## 1. Imports and real grid coordinates

In [1]:
import os
import numpy as np
import netCDF4 as nc
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = "compare_data"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

IM, JM, LM = 72, 46, 40  # P2SAoM40's real grid: lon, lat, atmosphere layers
SURFACE_LAYER = 0        # DRYCNV layer shown below

# Real lat/lon coordinates, from the archived P2SAoM40 production output
# (not derived from the synthetic benchmark data -- these are genuine).
_ref = nc.Dataset("ANN4099.aijP2SAoM40.nc")
lat = _ref.variables["lat"][:].data
lon = _ref.variables["lon"][:].data
_ref.close()

print(f"Real P2SAoM40 grid: {len(lon)} lon ({lon.min()}..{lon.max()}) x "
      f"{len(lat)} lat ({lat.min()}..{lat.max()})")

Real P2SAoM40 grid: 72 lon (-177.5..177.5) x 46 lat (-90.0..90.0)


## 2. Load data

Loads the outputs `compare_fortran.f90` / `compare_jax.py` /
`compare_run_gpu_interactive.py` already wrote to `compare_data/` (run those
first if this directory is empty — see `STATUS.md`).

In [2]:
def load_fortran_3d(field):
    path = os.path.join(DATA_DIR, f"drycnv_fortran_{field}_out.dat")
    return np.fromfile(path, dtype=np.float64).reshape((IM, JM, LM), order="F")

def load_fortran_1d(field):
    path = os.path.join(DATA_DIR, f"pbl_fortran_{field}_out.dat")
    return np.fromfile(path, dtype=np.float64)

def load_jax(module, device, field):
    path = os.path.join(DATA_DIR, f"{module}_jax_{device}_{field}_out.npy")
    return np.load(path) if os.path.exists(path) else None

def to_lat_lon(arr_im_jm):
    """(IM, JM) -> (JM, IM), i.e. (lat, lon), for plotting."""
    return np.asarray(arr_im_jm).T

DEVICES = ["cpu", "gpu"]
DRYCNV_FIELDS = ["T", "Q"]
PBL_FIELDS = ["u", "t", "q", "dpsim", "dpsih", "dpsiq"]

# --- DRYCNV: surface-layer (layer 0) slice of the 3D field ---
drycnv_maps = {}
for field in DRYCNV_FIELDS:
    fortran3d = load_fortran_3d(field)
    drycnv_maps[("fortran", field)] = to_lat_lon(fortran3d[:, :, SURFACE_LAYER])
    for dev in DEVICES:
        jax3d = load_jax("drycnv", dev, field)
        if jax3d is not None:
            drycnv_maps[(f"jax_{dev}", field)] = to_lat_lon(jax3d[:, :, SURFACE_LAYER])

# --- PBL: naturally single-layer ---
pbl_maps = {}
for field in PBL_FIELDS:
    flat_fortran = load_fortran_1d(field)
    pbl_maps[("fortran", field)] = to_lat_lon(flat_fortran.reshape(IM, JM))
    for dev in DEVICES:
        flat_jax = load_jax("pbl", dev, field)
        if flat_jax is not None:
            pbl_maps[(f"jax_{dev}", field)] = to_lat_lon(flat_jax.reshape(IM, JM))

print("DRYCNV legs/fields loaded:", sorted(drycnv_maps.keys()))
print("PBL legs/fields loaded:", sorted(pbl_maps.keys()))

DRYCNV legs/fields loaded: [('fortran', 'Q'), ('fortran', 'T'), ('jax_cpu', 'Q'), ('jax_cpu', 'T'), ('jax_gpu', 'Q'), ('jax_gpu', 'T')]
PBL legs/fields loaded: [('fortran', 'dpsih'), ('fortran', 'dpsim'), ('fortran', 'dpsiq'), ('fortran', 'q'), ('fortran', 't'), ('fortran', 'u'), ('jax_cpu', 'dpsih'), ('jax_cpu', 'dpsim'), ('jax_cpu', 'dpsiq'), ('jax_cpu', 'q'), ('jax_cpu', 't'), ('jax_cpu', 'u'), ('jax_gpu', 'dpsih'), ('jax_gpu', 'dpsim'), ('jax_gpu', 'dpsiq'), ('jax_gpu', 'q'), ('jax_gpu', 't'), ('jax_gpu', 'u')]


## 3. Plotting helpers

In [3]:
from IPython.display import display, HTML

_plotlyjs_loaded = {"done": False}

def show(fig):
    """Display a figure as self-contained HTML (Plotly.js inlined once, then
    reused) so it renders in any notebook frontend -- VS Code, JupyterLab,
    nbviewer, GitHub -- without depending on a live Plotly mimetype renderer
    extension being active. Same pattern already used in
    mantle/executive_summary_dashboard.ipynb."""
    include_js = "inline" if not _plotlyjs_loaded["done"] else False
    _plotlyjs_loaded["done"] = True
    html = fig.to_html(full_html=False, include_plotlyjs=include_js, config={"displayModeBar": False})
    display(HTML(html))

FIELD_COLORSCALE = {
    "T": "RdYlBu_r", "Q": "Blues",
    "u": "RdBu", "t": "RdYlBu_r", "q": "Blues",
    "dpsim": "RdBu", "dpsih": "RdBu", "dpsiq": "RdBu",
}

def create_2d_global_map(data, lat, lon, title, colorscale="Viridis", zmid=None):
    """2D global map (latitude vs. longitude) using Plotly -- same style as
    mantle/visualize_2d_global_maps.ipynb's create_2d_global_map."""
    kwargs = dict(z=data, x=lon, y=lat, colorscale=colorscale,
                  colorbar=dict(title=title), hoverongaps=False)
    if zmid is not None:
        kwargs["zmid"] = zmid
    fig = go.Figure(data=go.Heatmap(**kwargs))
    fig.update_layout(title=title, xaxis_title="Longitude", yaxis_title="Latitude",
                       width=800, height=400)
    return fig

LEG_LABELS = [("fortran", "Fortran (ifort -O2)"), ("jax_cpu", "JAX-CPU"), ("jax_gpu", "JAX-GPU")]

def leg_dashboard(maps, field, colorscale, title_prefix):
    """Side-by-side Fortran / JAX-CPU / JAX-GPU maps for one field."""
    available = [(k, l) for k, l in LEG_LABELS if (k, field) in maps]
    fig = make_subplots(rows=1, cols=len(available), subplot_titles=[l for _, l in available])
    for i, (k, l) in enumerate(available, start=1):
        fig.add_trace(
            go.Heatmap(z=maps[(k, field)], x=lon, y=lat, colorscale=colorscale,
                       showscale=(i == len(available))),
            row=1, col=i,
        )
    fig.update_layout(title=f"{title_prefix}.{field}: Fortran vs. JAX (CPU/GPU)",
                       height=350, width=350 * len(available))
    return fig

## 4. DRYCNV — surface layer (layer 0), Fortran vs. JAX (CPU/GPU)

In [4]:
drycnv_dashboards = {}
for field in DRYCNV_FIELDS:
    fig = leg_dashboard(drycnv_maps, field, FIELD_COLORSCALE[field], "drycnv")
    drycnv_dashboards[field] = fig
    show(fig)

## 5. PBL similarity — Fortran vs. JAX (CPU/GPU)

In [5]:
pbl_dashboards = {}
for field in PBL_FIELDS:
    fig = leg_dashboard(pbl_maps, field, FIELD_COLORSCALE[field], "pbl")
    pbl_dashboards[field] = fig
    show(fig)

## 6. Difference maps (JAX − Fortran)

A uniformly pale (near-zero) map here is the spatial confirmation of the
floating-point-level agreement already reported numerically in `STATUS.md`
(max abs diff ≤1.5e-3 CPU, ≤2.8e-3 GPU) — not evidence of
real physical structure, since the underlying field values are synthetic.

In [6]:
def diff_map(maps, field, device):
    ref_key, jax_key = ("fortran", field), (f"jax_{device}", field)
    if ref_key not in maps or jax_key not in maps:
        return None
    return maps[jax_key] - maps[ref_key]

diff_figs = {}
for group_name, maps, fields in [("drycnv", drycnv_maps, DRYCNV_FIELDS), ("pbl", pbl_maps, PBL_FIELDS)]:
    for field in fields:
        for dev in DEVICES:
            d = diff_map(maps, field, dev)
            if d is None:
                continue
            fig = create_2d_global_map(
                d, lat, lon,
                f"{group_name}.{field}: JAX-{dev.upper()} minus Fortran",
                colorscale="RdBu", zmid=0,
            )
            diff_figs[(group_name, field, dev)] = fig
            show(fig)
            print(f"{group_name}.{field}  jax_{dev} vs fortran  max|diff| = {np.max(np.abs(d)):.3e}")

drycnv.T  jax_cpu vs fortran  max|diff| = 7.833e-05


drycnv.T  jax_gpu vs fortran  max|diff| = 9.910e-05


drycnv.Q  jax_cpu vs fortran  max|diff| = 3.059e-09


drycnv.Q  jax_gpu vs fortran  max|diff| = 3.059e-09


pbl.u  jax_cpu vs fortran  max|diff| = 4.412e-04


...

pbl.dpsim  jax_gpu vs fortran  max|diff| = 1.088e-03


pbl.dpsih  jax_cpu vs fortran  max|diff| = 8.135e-04


pbl.dpsih  jax_gpu vs fortran  max|diff| = 1.140e-03


pbl.dpsiq  jax_cpu vs fortran  max|diff| = 1.258e-03


pbl.dpsiq  jax_gpu vs fortran  max|diff| = 1.092e-03


## 7. Save all maps as HTML

Saved into `outputs/`, prefixed `p2saom40_kernel_` to distinguish from the
older (currently broken) synthetic single-layer outputs already in that
directory.

In [7]:
saved = 0
for (leg, field), grid in {**drycnv_maps, **pbl_maps}.items():
    fig = create_2d_global_map(grid, lat, lon, f"{leg}: {field}",
                                colorscale=FIELD_COLORSCALE.get(field, "Viridis"))
    fig.write_html(os.path.join(OUT_DIR, f"p2saom40_kernel_{leg}_{field}_2d_map.html"))
    saved += 1

for (group, field, dev), fig in diff_figs.items():
    fig.write_html(os.path.join(OUT_DIR, f"p2saom40_kernel_{group}_{field}_diff_jax_{dev}_2d_map.html"))
    saved += 1

print(f"Saved {saved} HTML maps to {OUT_DIR}/")

Saved 40 HTML maps to outputs/


## Summary

- Grid and lat/lon axes: real (P2SAoM40's actual 72×46 medium-resolution grid).
- Field values: synthetic random test vectors, not real simulated climate —
  don't read physical meaning into the spatial patterns above.
- What this notebook actually demonstrates: JAX (CPU and real-GPU-measured)
  matches the real compiled Fortran reference to floating-point precision,
  spatially as well as in aggregate.
- Full numeric accuracy/timing table and sourcing: `STATUS.md`,
  `compare_data/summary.json`.